# HAMV-CC — Large-Dataset Reproducible Research Notebook

**Scalable research edition derived from the supplied `HAMV_CC_InDepth_Reproducible_Research_Notebook_Enhanced.ipynb`.**

This edition keeps the source methodology—source manifesting, 16-D handcrafted features, optional ResNet50 visual features, standardization/PCA, feature-space NT-Xent SSL, clustering, multi-seed stability, bootstrap stability, and evidence reporting—but changes the execution model for large archives.

### Large-dataset safeguards

- **No `IMAGES` list containing the whole archive.** Images are decoded one at a time or in bounded batches.
- **Disk-backed NumPy memmaps** are used for feature matrices.
- **ResNet50 inference is batched** and writes directly to disk.
- **MiniBatchKMeans** is the default clustering implementation.
- **Silhouette / CH / DB diagnostics are sampled** when the archive exceeds the configured sample limit.
- **Co-association matrices are capped** at a configurable document count because an \(N^2\) matrix is intrinsically non-scalable.
- **SSL uses minibatches** and does not put the full dataset on GPU.
- **Manifest, config, environment, errors, assignments, and summaries are persisted.**
- Exact duplicate SHA-256 groups remain visible rather than being silently removed.

> A large dataset run should be treated as a new experiment. Existing reported values from the original notebook are not reused as results for the new archive.


## 1. Research contract

Freeze these before a confirmatory run:

1. dataset root and manifest;
2. preprocessing switches;
3. feature ordering;
4. ResNet/PCA settings;
5. candidate `k` range;
6. seeds;
7. bootstrap and sampling limits;
8. representation/algorithm selection rule.

The pipeline records failed file reads in `cache/feature_errors.csv` rather than silently dropping them.

In [ ]:
# 2. Imports and configuration
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd

# Run this notebook from the package root.
sys.path.insert(0, str(Path.cwd()))

from hamvcc_research.config import Config
from hamvcc_research.provenance import build_manifest, write_environment
from hamvcc_research.features import extract_handcrafted, FEATURE_NAMES
from hamvcc_research.representations import resnet50_features, build_base_representation
from hamvcc_research.clustering import select_k, multi_seed_stability, bootstrap_stability, fit_kmeans, safe_internal_metrics

CONFIG_PATH = Path("configs/large_dataset.json")
CFG = Config.from_json(CONFIG_PATH)
OUT = Path(CFG.output_root)
OUT.mkdir(parents=True, exist_ok=True)
print(CFG)


In [ ]:
# 3. Provenance first: manifest before feature extraction
write_environment(OUT / "environment.json")
manifest = build_manifest(Path(CFG.dataset_root), CFG.image_extensions, OUT / "manifest.csv")
print("Dataset root:", Path(CFG.dataset_root).resolve())
print("Documents:", len(manifest))
print("Total GB:", manifest["bytes"].sum()/1e9 if len(manifest) else 0)
if len(manifest):
    print(manifest["ext"].value_counts())
    dup = manifest.groupby("sha256").size()
    print("Duplicate SHA-256 groups:", int((dup > 1).sum()))
manifest.head()


In [ ]:
# 4. Streaming handcrafted extraction
# The resulting array is a disk-backed .npy memmap.
cache = OUT / "cache"
cache.mkdir(exist_ok=True)
hand_path = cache / "handcrafted.npy"
hand, errors = extract_handcrafted(
    manifest, CFG.__dict__, hand_path, cache / "feature_errors.csv"
)
print("Feature matrix:", hand.shape)
print("Failed files:", len(errors))
pd.DataFrame(hand, columns=FEATURE_NAMES).describe().T[["mean","std","min","max"]]


In [ ]:
# 5. Feature diagnostics — constant columns are explicitly reported
hand_df = pd.DataFrame(np.asarray(hand), columns=FEATURE_NAMES)
diagnostics = pd.DataFrame({
    "feature": FEATURE_NAMES,
    "variance": hand_df.var().values,
    "missing": hand_df.isna().sum().values
})
diagnostics["constant"] = diagnostics["variance"] <= 1e-12
display(diagnostics)


In [ ]:
# 6. Optional batched ResNet50 visual features
visual = None
if CFG.use_resnet50:
    visual_path = cache / "resnet50.npy"
    visual = resnet50_features(
        manifest, CFG.__dict__, visual_path,
        batch_size=CFG.batch_size, num_workers=CFG.num_workers
    )
    print("Visual features:", visual.shape)
else:
    print("ResNet50 disabled.")


In [ ]:
# 7. Base representation: standardization + optional PCA
base, scaler, pca = build_base_representation(
    hand, visual,
    CFG.pca_target if CFG.pca else None
)
np.save(cache / "base_representation.npy", base)
print("Base representation:", base.shape)


## 8. Feature-space self-supervised learning

This is the scalable counterpart of the source notebook's feature-space NT-Xent path. Training uses bounded minibatches; the complete learned representation is written after inference. The GPU never receives the full archive at once.

In [ ]:
# 8. Train feature-space SSL (optional)
from hamvcc_research.ssl import train_feature_ssl

if CFG.run_ssl:
    model_ssl, ssl_rep, ssl_history = train_feature_ssl(
        base,
        epochs=CFG.ssl_epochs,
        batch_size=CFG.ssl_batch_size,
        lr=CFG.ssl_lr,
        temp=CFG.ssl_temperature,
        proj_dim=CFG.ssl_projection_dim,
        noise=CFG.ssl_noise,
        seed=CFG.seeds[0],
    )
    np.save(cache / "ssl_representation.npy", ssl_rep)
    ssl_history.to_csv(OUT / "ssl_history.csv", index=False)
    print("SSL representation:", ssl_rep.shape)
    display(ssl_history.tail())
else:
    ssl_rep = None
    print("SSL disabled.")


In [ ]:
# 9. Representation comparison with sampled internal metrics
representations = {"base": base}
if ssl_rep is not None:
    representations["ssl"] = ssl_rep

rep_rows = []
for name, X in representations.items():
    k, kdf = select_k(X, CFG.cluster_k_min, CFG.cluster_k_max, seed=CFG.seeds[0])
    kdf.to_csv(OUT / f"k_diagnostics_{name}.csv", index=False)
    stability, _ = multi_seed_stability(X, k, CFG.seeds, CFG.__dict__)
    lab = fit_kmeans(X, k, CFG.seeds[0], CFG.__dict__)
    metrics = safe_internal_metrics(X, lab)
    boot_mean, boot_std, _ = bootstrap_stability(
        X, lab, k, CFG.__dict__, B=CFG.bootstrap_runs,
        frac=CFG.bootstrap_fraction, seed=CFG.seeds[0]
    )
    rep_rows.append({
        "representation": name, "k": k,
        "seed_stability_ARI": stability,
        "bootstrap_ARI": boot_mean,
        "bootstrap_ARI_std": boot_std,
        **metrics
    })

REP_DF = pd.DataFrame(rep_rows)
REP_DF["rank"] = (
    REP_DF["silhouette"].rank(ascending=False)
    + REP_DF["DB"].rank(ascending=True)
    + REP_DF["seed_stability_ARI"].rank(ascending=False)
) / 3
REP_DF = REP_DF.sort_values("rank").reset_index(drop=True)
REP_DF.to_csv(OUT / "representation_comparison.csv", index=False)
display(REP_DF.round(4))


In [ ]:
# 10. Final clustering and reproducible assignment table
chosen = REP_DF.iloc[0]["representation"]
X_final = representations[chosen]
k_final = int(REP_DF.iloc[0]["k"])
labels = fit_kmeans(X_final, k_final, CFG.seeds[0], CFG.__dict__)

assign = pd.DataFrame({
    "index": np.arange(len(manifest)),
    "path": manifest["path"],
    "relative_path": manifest["relative_path"],
    "sha256": manifest["sha256"],
    "cluster": labels,
})
assign.to_csv(OUT / "final_assignments.csv", index=False)

print("Chosen representation:", chosen)
print("Final k:", k_final)
print("Internal metrics:", safe_internal_metrics(X_final, labels))
display(assign.head())


## 11. Scalable visualization policy

Do **not** create full \(N\times N\) co-association matrices, pairwise-distance matrices, or full-image montages for very large archives. The research package caps co-association at `max_coassociation_n` and should use sampled documents for visual audit.

For publication figures, record the sampling seed and sample size alongside the figure.

In [ ]:
# 12. Optional bounded co-association diagnostic
from hamvcc_research.clustering import coassociation
if len(X_final) <= CFG.max_coassociation_n:
    idx, A = coassociation(
        X_final, k_final, R=10,
        max_n=CFG.max_coassociation_n, seed=CFG.seeds[0]
    )
    np.save(OUT / "coassociation.npy", A)
    print("Co-association matrix:", A.shape)
else:
    print("Skipped: archive exceeds the configured N^2 diagnostic cap.")


In [ ]:
# 13. One-command complete run
# Uncomment to execute the exact same pipeline represented by the research scripts.
#
# from hamvcc_research.pipeline import run
# summary = run(CFG)
# print(json.dumps(summary, indent=2))


## 14. Evidence interpretation

- Internal clustering metrics are geometric diagnostics, not semantic truth.
- Automatic representation ranking is a convenience rule, not evidence that one representation is universally superior.
- Exact duplicate files should normally remain in the archive audit; their cluster consistency is a useful reproducibility check.
- Any change to preprocessing, feature construction, `k`, seeds, or selection rules creates a new experimental condition.
- For very large datasets, sampled diagnostics must report their sample size and seed.
